In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 30. ZeROtext FSDP — text text text

> **Learning Goals**
> - ZeRO Stage 1/2/3text text text text text
> - FSDP (Fully Sharded Data Parallel)text text text text
> - text text text text Calculationtext

## 30.1 DDPtext text text

DDP: text GPUtext Model text text. 70B Modeltext text GPUtext text text.

ZeROtext text: **"text text text"** — text text, text, text.

## 30.2 text text

Training text = text + text + text text + textValue

AdamW text:
- text $P$ (FP16): $2P$ bytes
- text (FP16): $2P$ bytes
- text text (FP32: m, v, master): $12P$ bytes
- text (textValue text): $16P$ bytes

7B Model: 112 GB (text GPU text)

## 30.3 ZeRO Stage 1 — text text text

text text $N$text GPUtext text:
- text: $16P / N + 4P$ (text + text text)
- 7B, N=8: $16P/8 + 4P = 2P + 4P = 6P = 42GB$ (text text)

## 30.4 ZeRO Stage 2 — textdegrees text

textdegrees text:
- text: $16P/N + 2P = 4P$ (8B + 4P + 4P)
- Reduce-Scatter text (All-Reduce text)

## 30.5 ZeRO Stage 3 — text text

text text:
- text: $16P/N$ + textValue
- 7B, N=8: $2P + text ≈ 14GB$ → text 40GB GPU text
- text: All-Gather (Forward Pass), Reduce-Scatter (Backpropagation)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_memory_gb(model_params_b, n_gpus, stage, optimizer_factor=12, dtype_bytes=2):
    """ZeRO stagetext Memory (GB).
    model_params_b: Parameter Count (text: 10text)
    optimizer_factor: Adamtext text 12 (m, v, master FP32)
    dtype_bytes: 2 for FP16
    """
    P = model_params_b  # 10text text
    # FP16 text: 2 bytes, FP16 grad: 2 bytes
    # Optimizer (FP32 m, v, master): 12 bytes per param
    param_mem = P * dtype_bytes  # GB
    grad_mem = P * dtype_bytes
    opt_mem = P * optimizer_factor

    if stage == 0:  # DDP
        return param_mem + grad_mem + opt_mem
    elif stage == 1:  # Optimizertext text
        return param_mem + grad_mem + opt_mem / n_gpus
    elif stage == 2:  # Optimizer + Gradient text
        return param_mem + (grad_mem + opt_mem) / n_gpus
    elif stage == 3:  # text text
        return (param_mem + grad_mem + opt_mem) / n_gpus

# Visualization: 7B Model, GPU text
model_size = 7  # 7B
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# text: GPU text
ax = axes[0]
for stage, label, color in [(0, 'DDP', 'blue'), (1, 'ZeRO-1', 'green'),
                             (2, 'ZeRO-2', 'orange'), (3, 'ZeRO-3', 'red')]:
    mems = [zero_memory_gb(model_size, n, stage) for n in n_gpus_list]
    ax.plot(n_gpus_list, mems, 'o-', label=label, linewidth=2, color=color)
ax.axhline(40, color='gray', linestyle='--', label='A100 40GB')
ax.axhline(80, color='gray', linestyle=':', label='A100 80GB')
ax.set_xlabel('GPU text')
ax.set_ylabel('GPU text Memory (GB)')
ax.set_title(f'ZeRO Stagetext Memory (7B Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# text: Model Sizetext
ax = axes[1]
for stage, label, color in [(0, 'DDP', 'blue'), (3, 'ZeRO-3', 'red')]:
    for n in [1, 8, 32]:
        sizes = [1, 7, 13, 30, 70, 175]
        mems = [zero_memory_gb(s, n, stage) for s in sizes]
        ax.plot(sizes, mems, 'o-', label=f'{label}, N={n}', linewidth=2)
ax.set_xlabel('Model Size (B params)')
ax.set_ylabel('GPU text Memory (GB)')
ax.set_title('Model Sizetext Memory')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch30_zero_memory.png', dpi=100, bbox_inches='tight')
plt.show()

# text Output
print(f"\n{'Model':>6} | {'GPU':>4} | {'DDP':>8} | {'ZeRO-1':>8} | {'ZeRO-2':>8} | {'ZeRO-3':>8}")
print("-" * 55)
for size in [7, 13, 70]:
    for n in [1, 8]:
        ddp = zero_memory_gb(size, n, 0)
        z1 = zero_memory_gb(size, n, 1)
        z2 = zero_memory_gb(size, n, 2)
        z3 = zero_memory_gb(size, n, 3)
        print(f"{size:>5}B | {n:>4} | {ddp:>7.1f}G | {z1:>7.1f}G | {z2:>7.1f}G | {z3:>7.1f}G")


## 30.6 PyTorch FSDP

FSDP (Fully Sharded Data Parallel) = PyTorch text ZeRO-3.

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyLargeModel()
model = FSDP(model)  # text text text
```

text:
1. Forward Pass text: All-Gathertext text text
2. Forward Pass text: text text text
3. Backpropagation: text All-Gather, text Calculation
4. Reduce-Scattertext text text

## 30.7 CPU Offloading

GPU text text text text text CPU RAMtext text.
- ZeRO-Infinity, FSDP with CPU offload
- Speedtext text text Model Training text


In [ ]:
# FSDP text (gradient sharding)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FSDPSimulator:
    """text GPUtext FSDPtext text text."""
    def __init__(self, model, n_shards=4):
        self.model = model
        self.n_shards = n_shards
        # text n_shardstext text (text "GPU"text text text)
        params = list(model.parameters())
        self.shards = [params[i::n_shards] for i in range(n_shards)]

    def step(self, x, y, optimizer):
        """text Step: Forward Pass → Backward Pass → text shardtext Update."""
        optimizer.zero_grad()
        out = self.model(x)
        loss = F.mse_loss(out, y)
        loss.backward()

        # text shardtext Gradienttext text (text shardtext 0text)
        # text FSDPtext text text, text text Gradient text
        optimizer.step()
        return loss.item()

# text
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 128)
)
fsdp = FSDPSimulator(model, n_shards=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(8, 128)
y = torch.randn(8, 128)
loss = fsdp.step(x, y, opt)
print(f"FSDP text loss: {loss:.4f}")
print(f"text text: {sum(p.numel() for p in model.parameters()):,}")
print(f"Shard text: {fsdp.n_shards} (text shardtext ~{sum(p.numel() for p in model.parameters()) // 4:,} text)")


## 30.8 Key Takeaways

| Stage | text text | text | text |
|---|---|---|---|
| DDP | text | $16P$ | All-Reduce |
| ZeRO-1 | text | $16P/N + 4P$ | All-Reduce |
| ZeRO-2 | + text | $16P/N + 2P$ | Reduce-Scatter |
| ZeRO-3 | + text | $16P/N$ | All-Gather + Reduce-Scatter |
| FSDP | ZeRO-3 (PyTorch) | $16P/N$ | text |
| ZeRO-Infinity | + CPU/NVMe | text text | text |

## Exercises

1. ZeRO Stage 0/1/2/3text text 70B Model, N=16text text Calculationtext.
2. FSDPtext Forward Pass text All-Gathertext text text text.
3. ZeRO-3text text text DDPtext Comparisontext.
4. CPU Offloadingtext Speedtext text text text text.
5. 70B Modeltext Trainingtext ZeRO-3 + text text A100 80GBtext text Calculationtext.

> Solutions: `solutions/ch30_solutions.ipynb`
